# Тема 02. Линейная регрессия и метрики качества
**Датасет:** California Housing (`MedHouseVal` — медианная стоимость дома в сотнях тысяч долларов, \$100k).  
**Цель работы:** пройти полный базовый пайплайн задачи регрессии — от первичного аудита данных, разбиения на `train/test` и константного `baseline` до собственной реализации линейной регрессии и метрик на `NumPy`, обучения `LinearRegression` из `scikit-learn` и диагностики остатков.

## 1. Подготовка и первичный анализ данных (EDA)
1. Загрузим датасет (если в папке `data/train.csv` или `train.csv` лежит CSV из Kaggle Playground S3E1 — читаем его; иначе загружаем стандартный `fetch_california_housing` из `sklearn.datasets`).
2. Изучим таблицу через `head()`, `info()`, `describe()`.
3. Проверим типы столбцов, пропуски, полные дубликаты и распределение целевой переменной `MedHouseVal`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams["figure.figsize"] = (10, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Автоматическая загрузка: локальный CSV с Kaggle S3E1 или встроенный California Housing
csv_candidates = [Path("data/train.csv"), Path("train.csv"), Path.home() / "Downloads" / "train.csv"]
csv_path = next((p for p in csv_candidates if p.exists()), None)

if csv_path is not None:
    df_raw = pd.read_csv(csv_path)
    print(f"Загружен локальный файл: {csv_path} ({df_raw.shape[0]} строк, {df_raw.shape[1]} столбцов)")
else:
    cal = fetch_california_housing(as_frame=True)
    df_raw = cal.frame.copy()
    print(f"Загружен датасет fetch_california_housing ({df_raw.shape[0]} строк, {df_raw.shape[1]} столбцов)")

df = df_raw.copy()
# В Kaggle Playground S3E1 есть технический столбец id — удаляем его, чтобы не нести шум в модель
if "id" in df.columns:
    df = df.drop(columns=["id"])
    print("Технический столбец 'id' удалён.")

df.head()

In [ ]:
# Информация о типах данных и памяти
df.info()

In [ ]:
# Описательные статистики по всем признакам и целевой переменной
df.describe().T

In [ ]:
# Проверка пропусков, типов данных и полных дубликатов
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_cnt": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(),
})
print(f"Полных дубликатов строк: {df.duplicated().sum()}")
audit

In [ ]:
# Распределение целевой переменной MedHouseVal
target_col = "MedHouseVal"
cap_val = df[target_col].max()
cap_share = (df[target_col] >= 4.999).mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

# 1. Гистограмма
axes[0].hist(df[target_col], bins=50, color="#0072B2", edgecolor="white", alpha=0.85)
axes[0].axvline(df[target_col].mean(), color="#D55E00", linestyle="--", linewidth=2, label=f"Среднее = {df[target_col].mean():.2f}")
axes[0].axvline(df[target_col].median(), color="#009E73", linestyle="-", linewidth=2, label=f"Медиана = {df[target_col].median():.2f}")
axes[0].set_title("Распределение целевой переменной MedHouseVal")
axes[0].set_xlabel("MedHouseVal ($100k)")
axes[0].set_ylabel("Количество районов")
axes[0].legend()

# 2. Box plot
axes[1].boxplot(df[target_col], vert=False, patch_artist=True, boxprops=dict(facecolor="#56B4E9", alpha=0.7))
axes[1].set_title("Разброс и выбросы MedHouseVal")
axes[1].set_xlabel("MedHouseVal ($100k)")
axes[1].set_yticks([])

plt.show()
print(f"Максимум MedHouseVal = {cap_val:.5f}. Доля объектов на потолке (>= 5.0): {cap_share:.2f}%")

**Вывод по первичному анализу данных:**
- Все признаки и целевая переменная `MedHouseVal` имеют числовой тип (`float64`), пропусков и полных дубликатов нет.
- У целевой переменной `MedHouseVal` среднее выше медианы (распределение скошено вправо), а на правом краю (`MedHouseVal = 5.00001`, то есть \$500 000) виден искусственный пик (~4.7–5% выборки). Это результат цензурирования данных при сборе переписи: все более дорогие дома были записаны со значением `5.0`.
- В признаках `AveRooms`, `AveBedrms`, `Population` и `AveOccup` максимальные значения на порядки превышают 75-й перцентиль (`Q3`), что указывает на наличие сильных правых выбросов.

### Отделение признаков от `MedHouseVal` и разбиение на `train` и `test`

**Почему `test` нельзя использовать при обучении модели:**
- Тестовая выборка имитирует новые, ранее не виденные данные из будущего.
- Если использовать `test` при подборе весов модели, расчёте среднего/стандартного отклонения (`StandardScaler`), заполнении пропусков или отборе признаков, возникает **утечка данных (data leakage)**: информация о правильных ответах или распределении теста просачивается в модель.
- В результате метрика на `test` окажется искусственно завышенной (самообман), а при запуске на реальных новых данных качество упадёт.

In [ ]:
# Отделяем матрицу признаков X и вектор целевой переменной y
X = df.drop(columns=[target_col])
y = df[target_col].copy()

# Разбиваем на обучающую (80%) и тестовую (20%) выборки с фиксированным random_state
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")
print(f"Среднее y_train = {y_train.mean():.4f}, Среднее y_test = {y_test.mean():.4f}")

---
## 2. Метрики качества (NumPy vs sklearn) и Baseline

Реализуем метрики `MAE`, `MSE`, `RMSE` и `R²` вручную через `NumPy`, сверим их со стандартными функциями `sklearn.metrics`, а затем посчитаем метрики для константного **Baseline** (предсказание среднего значения `y_train` для всех объектов).

In [ ]:
# Реализация метрик регрессии вручную через NumPy
def mae_np(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_t, y_p = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_t - y_p)))

def mse_np(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_t, y_p = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    return float(np.mean((y_t - y_p) ** 2))

def rmse_np(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(mse_np(y_true, y_pred)))

def r2_np(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_t, y_p = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    ss_res = np.sum((y_t - y_p) ** 2)
    ss_tot = np.sum((y_t - np.mean(y_t)) ** 2)
    return float(1.0 - ss_res / ss_tot)

def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    # Проверяем совпадение наших NumPy-функций с функциями из sklearn
    mae_val = mae_np(y_true, y_pred)
    mse_val = mse_np(y_true, y_pred)
    rmse_val = rmse_np(y_true, y_pred)
    r2_val = r2_np(y_true, y_pred)

    assert np.isclose(mae_val, mean_absolute_error(y_true, y_pred)), "MAE не совпал со sklearn!"
    assert np.isclose(mse_val, mean_squared_error(y_true, y_pred)), "MSE не совпал со sklearn!"
    assert np.isclose(rmse_val, np.sqrt(mean_squared_error(y_true, y_pred))), "RMSE не совпал со sklearn!"
    assert np.isclose(r2_val, r2_score(y_true, y_pred)), "R2 не совпал со sklearn!"

    return {
        "MAE": mae_val,
        "MSE": mse_val,
        "RMSE": rmse_val,
        "R2": r2_val,
    }

# Словарь для хранения результатов всех моделей
results: dict[str, dict[str, float]] = {}

# Константный Baseline: всегда предсказывает среднее по y_train
train_mean_target = float(y_train.mean())
y_pred_base_train = np.full(shape=len(y_train), fill_value=train_mean_target, dtype=float)
y_pred_base_test = np.full(shape=len(y_test), fill_value=train_mean_target, dtype=float)

results["1. Baseline (mean y_train) [train]"] = evaluate_predictions(y_train, y_pred_base_train)
results["1. Baseline (mean y_train) [test]"] = evaluate_predictions(y_test, y_pred_base_test)

pd.DataFrame(results).T.round(4)

**Пояснение по Baseline:**
- На обучающей выборке `R²` константного прогноза среднего равен ровно `0.0000`, а на тестовой — чуть меньше нуля (`~ -0.0002`), так как среднее `y_train` минимально отличается от среднего `y_test`.
- В среднем константный прогноз промахивается на `MAE ≈ 0.91` (\$91 000), а с учётом штрафа за крупные ошибки — на `RMSE ≈ 1.14` (\$114 000). Любая осмысленная модель обязана заметно снизить `MAE/RMSE` и поднять `R²` существенно выше нуля.

---
## 3. Линейная регрессия

### 3.1. Одномерная линейная регрессия вручную через NumPy (`MedInc` → `MedHouseVal`)
По матрице корреляций с целевой переменной выберем самый сильный числовой признак — медианный доход жителей района `MedInc`.
Для модели $\hat{y} = w \cdot x + b$ аналитическое решение метода наименьших квадратов (МНК):
$$w = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n}(x_i - \bar{x})^2}, \qquad b = \bar{y} - w\bar{x}$$

In [ ]:
# Посмотрим корреляцию всех признаков с целевой переменной на обучающей выборке
corrs = X_train.apply(lambda col: col.corr(y_train)).sort_values(ascending=False)
print("Корреляция Пирсона признаков с MedHouseVal (на train):")
print(corrs.round(4))

In [ ]:
# Ручная реализация одномерной линейной регрессии через NumPy по признаку MedInc
feature_1d = "MedInc"
x_tr_1d = X_train[feature_1d].to_numpy(dtype=float)
x_te_1d = X_test[feature_1d].to_numpy(dtype=float)
y_tr_np = y_train.to_numpy(dtype=float)

x_mean = np.mean(x_tr_1d)
y_mean = np.mean(y_tr_np)

w_manual = np.sum((x_tr_1d - x_mean) * (y_tr_np - y_mean)) / np.sum((x_tr_1d - x_mean) ** 2)
b_manual = y_mean - w_manual * x_mean

# Предсказания ручной модели
y_pred_1d_train = w_manual * x_tr_1d + b_manual
y_pred_1d_test = w_manual * x_te_1d + b_manual

# Сверка с LinearRegression из sklearn на одном признаке
lr_1d_check = LinearRegression().fit(X_train[[feature_1d]], y_train)
assert np.isclose(w_manual, lr_1d_check.coef_[0]), "Коэффициент w не совпал со sklearn!"
assert np.isclose(b_manual, lr_1d_check.intercept_), "Свободный член b не совпал со sklearn!"

results["2. Manual 1D NumPy (MedInc) [train]"] = evaluate_predictions(y_train, y_pred_1d_train)
results["2. Manual 1D NumPy (MedInc) [test]"] = evaluate_predictions(y_test, y_pred_1d_test)

print(f"Формула одномерной модели: MedHouseVal = {w_manual:.4f} * MedInc + {b_manual:.4f}")
pd.DataFrame(results).T.round(4)

In [ ]:
# Визуализация точек тестовой выборки, константного baseline и найденной прямой регрессии
fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)

# Чтобы график был читаемым, отрисуем подвыборку из 1500 точек теста
sample_idx = np.random.choice(len(x_te_1d), size=min(1500, len(x_te_1d)), replace=False)
ax.scatter(
    x_te_1d[sample_idx],
    y_test.to_numpy()[sample_idx],
    alpha=0.25,
    s=18,
    color="#0072B2",
    label="Объекты test (подвыборка)",
)

x_grid = np.linspace(x_te_1d.min(), x_te_1d.max(), 200)
ax.plot(
    x_grid,
    w_manual * x_grid + b_manual,
    color="#D55E00",
    linewidth=2.5,
    label=f"Линейная регрессия: ŷ = {w_manual:.2f}·x + {b_manual:.2f}",
)
ax.axhline(
    train_mean_target,
    color="#009E73",
    linestyle="--",
    linewidth=2,
    label=f"Baseline (ȳ_train = {train_mean_target:.2f})",
)

ax.set_title("Одномерная линейная регрессия (MedInc → MedHouseVal)")
ax.set_xlabel("Медианный доход района MedInc ($10k)")
ax.set_ylabel("Медианная стоимость дома MedHouseVal ($100k)")
ax.legend()
plt.show()

### 3.2. Многомерная `LinearRegression` из `sklearn` на всех числовых признаках
Обучим `LinearRegression` на всех 8 числовых признаках. Дополнительно посчитаем коэффициенты на масштабированных признаках (`StandardScaler`, обученный строго на `X_train`), чтобы честно сравнить силу вклада признаков разного масштаба.

In [ ]:
# Обучаем LinearRegression на всех числовых признаках
lr_full = LinearRegression()
lr_full.fit(X_train, y_train)

y_pred_train = lr_full.predict(X_train)
y_pred_test = lr_full.predict(X_test)

# Дополнительно обучим на стандартизированных признаках (только для честного сравнения величины весов)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
lr_scaled = LinearRegression().fit(X_train_scaled, y_train)

coef_table = pd.DataFrame({
    "feature": X_train.columns,
    "coef_raw": lr_full.coef_,
    "coef_std (StandardScaler)": lr_scaled.coef_,
    "sign": np.where(lr_full.coef_ >= 0, "+ (растёт)", "- (падает)"),
}).sort_values(by="coef_std (StandardScaler)", key=np.abs, ascending=False)

print(f"Свободный член (intercept_) на сырых признаках: {lr_full.intercept_:.4f}")
print(f"Свободный член (intercept_) после StandardScaler: {lr_scaled.intercept_:.4f} (= y_train.mean())")
coef_table.round(5)

In [ ]:
# Наглядный график стандартизированных коэффициентов модели
fig, ax = plt.subplots(figsize=(8.5, 4.5), constrained_layout=True)
coef_plot = coef_table.sort_values(by="coef_std (StandardScaler)", ascending=True)
colors = ["#D55E00" if v < 0 else "#009E73" for v in coef_plot["coef_std (StandardScaler)"]]

ax.barh(coef_plot["feature"], coef_plot["coef_std (StandardScaler)"], color=colors, alpha=0.88)
ax.axvline(0, color="#1E2433", linewidth=1.2)
ax.set_title("Вклад признаков после масштабирования (StandardScaler)")
ax.set_xlabel("Изменение MedHouseVal ($100k) при сдвиге признака на 1 стандартное отклонение")
plt.show()

**Интерпретация знаков и величины коэффициентов:**
1. **`MedInc` (`+`)** — главный драйвер цены: чем выше медианный доход жителей квартала, тем дороже недвижимость.
2. **`Latitude` (`-`) и `Longitude` (`-`)** — географический градиент Калифорнии: при смещении на север и вглубь материка (прочь от прибрежных агломераций Лос-Анджелеса и залива Сан-Франциско) жильё дешевеет.
3. **`HouseAge` (`+`)** — более старые кварталы часто расположены в исторически развитых центрах городов у побережья, поэтому коэффициент умеренно положительный.
4. **Парадокс `AveRooms` (`-`) и `AveBedrms` (`+`)** — классический пример **мультиколлинеарности**: общее число комнат и число спален сильно коррелируют между собой. Когда число спален `AveBedrms` зафиксировано, рост общего числа комнат снижает оценку, и наоборот. Поэтому интерпретировать такие скоррелированные признаки по отдельности нужно с осторожностью.
5. **Влияние масштаба:** на сырых данных коэффициент у `AveBedrms` кажется огромным, а у `Population` — почти нулевым просто потому, что население измеряется тысячами человек, а спальни — единицами. После `StandardScaler` видно, что реальный вклад `Population` и `AveOccup` мал, а сильнее всего работают координаты и `MedInc`.

---
## 4. Метрики качества и анализ ошибок

In [ ]:
# Считаем метрики полной модели на train и test (наша функция внутри сверяет NumPy и sklearn)
results["3. Sklearn LinearRegression (all features) [train]"] = evaluate_predictions(y_train, y_pred_train)
results["3. Sklearn LinearRegression (all features) [test]"] = evaluate_predictions(y_test, y_pred_test)

summary_df = pd.DataFrame(results).T
# Посчитаем улучшение полной модели на test относительно Baseline
base_test = summary_df.loc["1. Baseline (mean y_train) [test]"]
full_test = summary_df.loc["3. Sklearn LinearRegression (all features) [test]"]

print("Сравнение моделей (NumPy == sklearn проверено через assert):")
print(
    f"На test относительно Baseline: "
    f"MAE снизился на {(1 - full_test['MAE'] / base_test['MAE']) * 100:.1f}%, "
    f"RMSE снизился на {(1 - full_test['RMSE'] / base_test['RMSE']) * 100:.1f}%, "
    f"R2 вырос с {base_test['R2']:.4f} до {full_test['R2']:.4f}."
)
summary_df.round(4)

In [ ]:
# Диагностические графики: 1) Предсказание vs Истина, 2) Распределение остатков, 3) Остатки vs Предсказание
residuals_test = y_test.to_numpy() - y_pred_test

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)

# 1. Истина против предсказания
axes[0].scatter(y_test, y_pred_test, alpha=0.2, s=14, color="#0072B2")
lims = [0, 5.5]
axes[0].plot(lims, lims, color="#D55E00", linestyle="--", linewidth=2, label="Идеал (ŷ = y)")
axes[0].set_xlim(lims)
axes[0].set_ylim(-0.5, 6.5)
axes[0].set_title("1. Истинные значения vs Предсказания")
axes[0].set_xlabel("Истинный MedHouseVal ($100k)")
axes[0].set_ylabel("Предсказанный ŷ ($100k)")
axes[0].legend()

# 2. Гистограмма остатков (y - ŷ)
axes[1].hist(residuals_test, bins=60, color="#009E73", edgecolor="white", alpha=0.85)
axes[1].axvline(0, color="#D55E00", linestyle="--", linewidth=2, label="Нулевая ошибка")
axes[1].set_title("2. Распределение остатков (y − ŷ)")
axes[1].set_xlabel("Остаток y − ŷ ($100k)")
axes[1].set_ylabel("Количество объектов")
axes[1].legend()

# 3. Остатки против предсказаний
axes[2].scatter(y_pred_test, residuals_test, alpha=0.2, s=14, color="#CC79A7")
axes[2].axhline(0, color="#D55E00", linestyle="--", linewidth=2)
axes[2].set_xlim(-0.5, 6.5)
axes[2].set_ylim(-4.5, 4.5)
axes[2].set_title("3. Остатки против предсказаний")
axes[2].set_xlabel("Предсказанный ŷ ($100k)")
axes[2].set_ylabel("Остаток y − ŷ ($100k)")

plt.show()

In [ ]:
# Найдём топ-10 объектов из test с самой большой абсолютной ошибкой |y - ŷ|
error_df = X_test.copy()
error_df["y_true"] = y_test
error_df["y_pred"] = y_pred_test
error_df["residual (y - ŷ)"] = residuals_test
error_df["abs_error |y - ŷ|"] = np.abs(residuals_test)

top_errors = error_df.sort_values(by="abs_error |y - ŷ|", ascending=False).head(10)
print("Топ-10 объектов тестовой выборки с максимальной абсолютной ошибкой:")
top_errors.round(3)

In [ ]:
# Сравним медианы признаков по всей выборке со значениями в топ-10 ошибок
comparison = pd.DataFrame({
    "train_median": X_train.median(),
    "train_q99": X_train.quantile(0.99),
    "top10_err_mean": top_errors[X_train.columns].mean(),
    "top10_err_max": top_errors[X_train.columns].max(),
})
comparison.round(2)

---
## 5. Итоговые выводы

### 1. Какую зависимость пытается найти линейная регрессия
Линейная регрессия ищет **линейную зависимость** между признаками и целевой переменной в виде взвешенной суммы $\hat{y} = w_1 x_1 + w_2 x_2 + \dots + w_k x_k + b$ (геометрически — прямую на плоскости или гиперплоскость в многомерном пространстве). Она предполагает, что при изменении признака на единицу ответ всегда меняется на одну и ту же величину $w_j$ независимо от значений остальных признаков.

### 2. Смысл каждой метрики и почему одной метрики недостаточно
- **MAE (Mean Absolute Error)** — средняя абсолютная ошибка в исходных единицах таргета (\$100k). Показывает, на сколько в среднем ошибается модель на типичном объекте, и устойчива к единичным выбросам.
- **MSE (Mean Squared Error)** — средний квадрат ошибки. Из-за возведения в квадрат непропорционально сильно штрафует за крупные промахи, но измеряется в квадратах единиц таргета.
- **RMSE (Root Mean Squared Error)** — корень из MSE. Возвращает ошибку в исходные единицы (\$100k), но сохраняет высокую чувствительность к выбросам. Если `RMSE` заметно выше `MAE` (у нас `0.74` против `0.53`), значит в данных есть объекты с очень крупными промахами.
- **$R^2$ (коэффициент детерминации)** — безразмерная доля дисперсии целевой переменной, объяснённая моделью по сравнению с константным прогнозом среднего (`Baseline` даёт $R^2 = 0$, идеальная модель — $R^2 = 1$).
- **Почему одной метрики мало:** `R²` показывает относительный выигрыш над средним, но не говорит о цене ошибки в долларах; `MAE` скрывает редкие катастрофические промахи, а `RMSE` их подсвечивает. Только связка `MAE + RMSE + R²` вместе с графиком остатков даёт полную картину.

### 3. Пять главных выводов о модели и её ошибках
1. **Уверенный выигрыш над Baseline:** константный прогноз среднего давал `MAE = 0.91`, `RMSE = 1.14`, `R² = 0.00`. Многомерная линейная регрессия снизила `MAE` до `~0.53` (ошибка ~\$53k), `RMSE` до `~0.74` и подняла `R²` до `~0.58–0.60`.
2. **Добавление остальных признаков к `MedInc` заметно помогает:** один признак `MedInc` объясняет около `46–47%` дисперсии (`R² ≈ 0.46`), а подключение географии (`Latitude`, `Longitude`) и характеристик жилья поднимает `R²` почти до `0.60`.
3. **Переобучения нет:** метрики на `train` и `test` практически совпадают (`RMSE ≈ 0.72–0.74`), так как у линейной регрессии всего 8 весов на десятки тысяч строк.
4. **Цензурирование таргета на уровне `5.0` искажает остатки:** на графике «Истина vs Предсказание» видна вертикальная стенка при `y = 5.0` (на графике остатков — наклонная полоса). Модель не знает про искусственный потолок \$500k и систематически занижает цену элитных районов.
5. **Экстремальные выбросы в признаках дают гигантские единичные промахи:** в топ-10 ошибок попадают объекты с аномальными значениями `AveOccup` (сотни человек на домохозяйство — например, общежития/казармы) или аномальным числом комнат `AveRooms`, из-за которых линейная формула улетает на несколько сотен тысяч долларов мимо цели.

### 4. Идеи для улучшения результата
1. **Обработка выбросов и скошенных признаков:** обрезать (`clip`) экстремальные квантили и применить `np.log1p` к правоскошенным признакам (`AveOccup`, `AveRooms`, `AveBedrms`, `Population`).
2. **Конструирование новых признаков (Feature Engineering):**
   - отношение спален к общему числу комнат (`bedroom_ratio = AveBedrms / AveRooms`) вместо двух коллинеарных столбцов;
   - число комнат на человека (`rooms_per_person = AveRooms / AveOccup`);
   - географические признаки: расстояние по координатам до центров Лос-Анджелеса, Сан-Франциско и береговой линии.
3. **Нелинейность и регуляризация:** добавить полиномиальные признаки (`PolynomialFeatures`) в связке с `StandardScaler` и регуляризацией (`Ridge` / `Lasso`), чтобы уловить нелинейные зависимости без взрыва весов.
4. **Переход к деревянным ансамблям:** использовать `RandomForestRegressor` или градиентный бустинг (`HistGradientBoostingRegressor` / `CatBoost` / `LightGBM`), которые естественным образом ловят нелинейные границы на карте Калифорнии и не боятся выбросов в `AveOccup`.